# QAOA-GPT inference demo

In [1]:
import pandas as pd
import networkx as nx
import random
from tqdm import tqdm

from src.model_interface import QAOA_GPT
from src.adapt_utils import compute_metrics 


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


## Loading a model

In [2]:
qaoa_gpt = QAOA_GPT(
    model_ckpt='nanoGPT/out-9_nodes_gnn/llama_ckpt_4500_gnn_ar_0_92291__er_0_0.pt',
    data_dir='nanoGPT/data/9_nodes_gnn', # to take meta.pkl file 
)


Model type: llama
Pool type: qaoa_double_pool
Embedding method: gnn
Number of nodes: 9


In [3]:
# qaoa_gpt = QAOA_GPT(
#     model_ckpt='nanoGPT/models/n10w_qaoa_mixer/ckpt_16000_gemb__ar_0_96584__er_0_0.pt',
#     data_dir='nanoGPT/models/n10w_qaoa_mixer/data', # to take meta.pkl file 
# )

## Generating random graphs

In [4]:
def add_weights_to_nx_graph(nx_graph):
    for u, v in nx_graph.edges():
        cur_weight = round(random.uniform(0, 1), 2)
        while cur_weight == 0:
            cur_weight = round(random.uniform(0, 1), 2)
        nx_graph[u][v]['weight'] = cur_weight
    return nx_graph

In [5]:
tqdm.pandas()

Modify this nodes here to match the model before run

In [6]:
n_graphs = 5
n_nodes = 9

In [7]:
graphs_edgelist_list_dict = dict()

er_graphs_edgelist_list_dict = dict()
for i in range(n_graphs):
    p = random.randrange(6,9) / 10
    cur_graph = nx.erdos_renyi_graph(
        n=n_nodes,
        p=p
    )
    
    er_graphs_edgelist_list_dict[f'er_graph_{i}'] = add_weights_to_nx_graph(cur_graph)

graphs_edgelist_list_dict.update(er_graphs_edgelist_list_dict)

In [8]:
graphs_edgelist_list_dict['er_graph_2'].edges(data=True)

EdgeDataView([(0, 1, {'weight': 0.77}), (0, 2, {'weight': 0.59}), (0, 3, {'weight': 0.65}), (0, 4, {'weight': 0.03}), (0, 6, {'weight': 0.7}), (0, 7, {'weight': 0.4}), (1, 2, {'weight': 0.14}), (1, 3, {'weight': 0.91}), (1, 4, {'weight': 0.06}), (1, 6, {'weight': 0.5}), (1, 7, {'weight': 0.18}), (2, 4, {'weight': 0.92}), (2, 5, {'weight': 0.61}), (2, 6, {'weight': 0.75}), (2, 7, {'weight': 0.14}), (2, 8, {'weight': 0.21}), (3, 4, {'weight': 0.56}), (3, 7, {'weight': 0.71}), (3, 8, {'weight': 0.21}), (4, 5, {'weight': 0.99}), (4, 7, {'weight': 0.6}), (4, 8, {'weight': 0.87}), (5, 6, {'weight': 0.62}), (5, 8, {'weight': 0.1}), (6, 7, {'weight': 0.99}), (6, 8, {'weight': 0.55})])

## Generate circuits with QAOA-GPT model

In [9]:
embedding_method = qaoa_gpt.embedding_method
print(f"Using embedding method: {embedding_method}")

qaoa_gpt_circ_df = qaoa_gpt.generate_circ_from_nx(
    graphs_edgelist_list_dict,
    # calculate_classic_maxcut=True, # to create col enery_gurobi. Default:True
    n_samples_per_batch=50, # max number of distinct graphs in a batch
    num_samples=5, # number of samples to draw
    max_new_tokens=150, # number of tokens generated in each sample
    temperature=0.1, # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
    top_k=200, # retain only the top_k most likely tokens, clamp others to have 0 probability
)

Using embedding method: gnn


Preparing graphs...:   0%|          | 0/5 [00:00<?, ?it/s]

Restricted license - for non-production use only - expires 2027-11-29


Preparing graphs...: 100%|██████████| 5/5 [00:00<00:00, 185.93it/s]


Performing gnn embedding
GNN model initialized on device: cuda


GNN:   0%|          | 0/5 [00:00<?, ?it/s]/home/mrzaizai2k/anaconda3/envs/adapt_gpt/lib/python3.10/site-packages/torch_geometric/utils/convert.py:278: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  data_dict[key] = torch.as_tensor(value)
GNN: 100%|██████████| 5/5 [00:00<00:00, 26.98it/s]


GNN shape: (5, 500)
graph_par_emb: [[0.08 0.01 0.56 ... 0.74 0.01 0.01]
 [0.01 0.01 0.94 ... 0.19 0.01 0.01]
 [0.01 0.01 0.01 ... 0.24 0.01 0.01]
 [0.22 0.22 0.01 ... 0.01 0.01 0.01]
 [0.01 0.01 0.75 ... 0.2  0.01 0.13]]
len(graph_par_emb[0]): 500


Inference. Current batch: n_edges: 23, n_graphs: 1: 100%|██████████| 5/5 [00:05<00:00,  1.02s/it]

gc_df:                                                                                                                                                                                                                                                                                                                                                                                                                   graph  \
0                               [(1, 3), 0.46, (1, 5), 0.39, (1, 6), 0.35, (2, 3), 0.76, (2, 4), 0.31, (2, 5), 0.53, (2, 6), 0.45, (2, 7), 0.31, (2, 9), 0.1, (3, 4), 0.46, (3, 5), 0.72, (3, 6), 0.34, (3, 7), 0.79, (3, 8), 0.4, (4, 5), 0.08, (4, 6), 0.24, (4, 7), 0.77, (4, 8), 0.59, (4, 9), 0.94, (5, 8), 0.29, (5, 9), 0.7, (6, 7), 0.45, (6, 8), 0.92, (6, 9), 0.82, (7, 8), 0.91, (7, 9), 0.14, (8, 9), 0.39]   
1  [(1, 2), 0.95, (1, 3), 0.6, (1, 4), 0.28, (1, 6), 0.36, (1, 8), 0.01, (1, 9), 0.23, (2, 3), 0.37, (2, 4), 0.27, (2, 5), 0.67, (2, 6), 0.64, (2, 7), 0.91, (2, 8), 0.66, 

In [10]:
sample_gr = graphs_edgelist_list_dict['er_graph_0'].edges(data=True)
sample_gr

EdgeDataView([(0, 2, {'weight': 0.46}), (0, 4, {'weight': 0.39}), (0, 5, {'weight': 0.35}), (1, 2, {'weight': 0.76}), (1, 3, {'weight': 0.31}), (1, 4, {'weight': 0.53}), (1, 5, {'weight': 0.45}), (1, 6, {'weight': 0.31}), (1, 8, {'weight': 0.1}), (2, 3, {'weight': 0.46}), (2, 4, {'weight': 0.72}), (2, 5, {'weight': 0.34}), (2, 6, {'weight': 0.79}), (2, 7, {'weight': 0.4}), (3, 4, {'weight': 0.08}), (3, 5, {'weight': 0.24}), (3, 6, {'weight': 0.77}), (3, 7, {'weight': 0.59}), (3, 8, {'weight': 0.94}), (4, 7, {'weight': 0.29}), (4, 8, {'weight': 0.7}), (5, 6, {'weight': 0.45}), (5, 7, {'weight': 0.92}), (5, 8, {'weight': 0.82}), (6, 7, {'weight': 0.91}), (6, 8, {'weight': 0.14}), (7, 8, {'weight': 0.39})])

In [11]:
len(graphs_edgelist_list_dict['er_graph_0'].edges(data=True))

27

The graph after prediction is shifted by 1 unit. For example, in NetworkX the edge is (0, 2, 0.36), but in this DataFrame it becomes (1, 3, 0.36)

In [12]:
qaoa_gpt_circ_df[:1]

,graph,n_edges,q_circuits,adapt_circuit,adapt_full_ar,graph_prefix,energy_gurobi,label,graph_w_jl,graph_weight_norm
0,"[(1, 3), 0.46, (1, 5), 0.39, (1, 6), 0.35, (2, 3), 0.76, (2, 4), 0.31, (2, 5), 0.53, (2, 6), 0.45, (2, 7), 0.31, (2, 9), 0.1, (3, 4), 0.46, (3, 5), 0.72, (3, 6), 0.34, (3, 7), 0.79, (3, 8), 0.4, (4, 5), 0.08, (4, 6), 0.24, (4, 7), 0.77, (4, 8), 0.59, (4, 9), 0.94, (5, 8), 0.29, (5, 9), 0.7, (6, 7), 0.45, (6, 8), 0.92, (6, 9), 0.82, (7, 8), 0.91, (7, 9), 0.14, (8, 9), 0.39]",27,"[[new_layer_p, 10, -0.5, 0.35, new_layer_p, 10, -0.37, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.14, 1.26], [new_layer_p, 10, -0.5, 0.35, new_layer_p, 10, -0.36, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.09, 1.26], [new_layer_p, 10, -0.5, 0.35, new_layer_p, 10, -0.37, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.14, 1.26, new_layer_p, 10, -0.09, 1.26], [new_layer_p, 10, -0.5, 0.35, new_layer_p, 10, -0.4, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 0.86, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.18, 1.13, new_layer_p, 10, -0.14, 1.26], [new_layer_p, 10, -0.5, 0.35, new_layer_p, 10, -0.36, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.14, 1.26]]",[],None,er_graph_0,-9.96,test_interactive,"[[1, 3, 0.46], [1, 5, 0.39], [1, 6, 0.35], [2, 3, 0.76], [2, 4, 0.31], [2, 5, 0.53], [2, 6, 0.45], [2, 7, 0.31], [2, 9, 0.1], [3, 4, 0.46], [3, 5, 0.72], [3, 6, 0.34], [3, 7, 0.79], [3, 8, 0.4], [4, 5, 0.08], [4, 6, 0.24], [4, 7, 0.77], [4, 8, 0.59], [4, 9, 0.94], [5, 8, 0.29], [5, 9, 0.7], [6, 7, 0.45], [6, 8, 0.92], [6, 9, 0.82], [7, 8, 0.91], [7, 9, 0.14], [8, 9, 0.39]]",1.0


In [13]:
qaoa_gpt_circ_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   graph              5 non-null      object 
 1   n_edges            5 non-null      int64  
 2   q_circuits         5 non-null      object 
 3   adapt_circuit      5 non-null      object 
 4   adapt_full_ar      0 non-null      object 
 5   graph_prefix       5 non-null      object 
 6   energy_gurobi      5 non-null      float64
 7   label              5 non-null      object 
 8   graph_w_jl         5 non-null      object 
 9   graph_weight_norm  5 non-null      float64
dtypes: float64(2), int64(1), object(7)
memory usage: 528.0+ bytes


## Evaluate circuits

In [14]:
qaoa_gpt_circ_eval_df = qaoa_gpt.eval_circ_df_jl(qaoa_gpt_circ_df)


===== DEBUG INFO =====
CWD: /home/mrzaizai2k/code_bao/ADAPT_GPT
Command:
/opt/julia-1.12.1/bin/julia -t 4 --project=/home/mrzaizai2k/code_bao/ADAPT_GPT/ADAPT.jl /home/mrzaizai2k/code_bao/ADAPT_GPT/adapt_gpt_eval_energy.jl /home/mrzaizai2k/code_bao/ADAPT_GPT/temp_data/adapt_gpt_res_2026-05-04__20_10_38_df.json /home/mrzaizai2k/code_bao/ADAPT_GPT/temp_data/adapt_gpt_res_2026-05-04__20_10_38_df_jl.json 9 qaoa_double_pool


Julia return code: 0


In [15]:
sample_gr

EdgeDataView([(0, 2, {'weight': 0.46}), (0, 4, {'weight': 0.39}), (0, 5, {'weight': 0.35}), (1, 2, {'weight': 0.76}), (1, 3, {'weight': 0.31}), (1, 4, {'weight': 0.53}), (1, 5, {'weight': 0.45}), (1, 6, {'weight': 0.31}), (1, 8, {'weight': 0.1}), (2, 3, {'weight': 0.46}), (2, 4, {'weight': 0.72}), (2, 5, {'weight': 0.34}), (2, 6, {'weight': 0.79}), (2, 7, {'weight': 0.4}), (3, 4, {'weight': 0.08}), (3, 5, {'weight': 0.24}), (3, 6, {'weight': 0.77}), (3, 7, {'weight': 0.59}), (3, 8, {'weight': 0.94}), (4, 7, {'weight': 0.29}), (4, 8, {'weight': 0.7}), (5, 6, {'weight': 0.45}), (5, 7, {'weight': 0.92}), (5, 8, {'weight': 0.82}), (6, 7, {'weight': 0.91}), (6, 8, {'weight': 0.14}), (7, 8, {'weight': 0.39})])

The eval df add 1 more col is adapt_gpt_energies

Each graph is generate into 5 other graphs, that why we see list of 5 q_circuits and adapt_gpt_energies



In [16]:
qaoa_gpt_circ_eval_df[:1]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_0,"[[1, 3], 0.46, [1, 5], 0.39, [1, 6], 0.35000000000000003, [2, 3], 0.76, [2, 4], 0.31, [2, 5], 0.53, [2, 6], 0.45, [2, 7], 0.31, [2, 9], 0.1, [3, 4], 0.46, [3, 5], 0.72, [3, 6], 0.34, [3, 7], 0.79, [3, 8], 0.4, [4, 5], 0.08, [4, 6], 0.24, [4, 7], 0.77, [4, 8], 0.59, [4, 9], 0.9400000000000001, [5, 8], 0.29, [5, 9], 0.7000000000000001, [6, 7], 0.45, [6, 8], 0.92, [6, 9], 0.8200000000000001, [7, 8], 0.91, [7, 9], 0.14, [8, 9], 0.39]",27,"[[new_layer_p, 10, -0.5, 0.35000000000000003, new_layer_p, 10, -0.37, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.14, 1.26], [new_layer_p, 10, -0.5, 0.35000000000000003, new_layer_p, 10, -0.36, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.09, 1.26], [new_layer_p, 10, -0.5, 0.35000000000000003, new_layer_p, 10, -0.37, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.14, 1.26, new_layer_p, 10, -0.09, 1.26], [new_layer_p, 10, -0.5, 0.35000000000000003, new_layer_p, 10, -0.4, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 0.86, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.18, 1.13, new_layer_p, 10, -0.14, 1.26], [new_layer_p, 10, -0.5, 0.35000000000000003, new_layer_p, 10, -0.36, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.14, 1.26]]","[-9.563416754115298, -9.543284963954012, -9.624512865359799, -9.667101152061932, -9.553682081589724]",-9.96


In [17]:
qaoa_gpt_circ_eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   graph_prefix        5 non-null      object 
 1   graph               5 non-null      object 
 2   n_edges             5 non-null      int64  
 3   q_circuits          5 non-null      object 
 4   adapt_gpt_energies  5 non-null      object 
 5   energy_gurobi       5 non-null      float64
dtypes: float64(1), int64(1), object(4)
memory usage: 368.0+ bytes


In [18]:
# 3 extract first rows into 5 rows for 5 q_circuits and adapt_gpt_energies
qaoa_gpt_circ_eval_expl_df = qaoa_gpt_circ_eval_df.explode(['adapt_gpt_energies', 'q_circuits']) 

In [19]:
qaoa_gpt_circ_eval_expl_df[:2]


,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_0,"[[1, 3], 0.46, [1, 5], 0.39, [1, 6], 0.35000000000000003, [2, 3], 0.76, [2, 4], 0.31, [2, 5], 0.53, [2, 6], 0.45, [2, 7], 0.31, [2, 9], 0.1, [3, 4], 0.46, [3, 5], 0.72, [3, 6], 0.34, [3, 7], 0.79, [3, 8], 0.4, [4, 5], 0.08, [4, 6], 0.24, [4, 7], 0.77, [4, 8], 0.59, [4, 9], 0.9400000000000001, [5, 8], 0.29, [5, 9], 0.7000000000000001, [6, 7], 0.45, [6, 8], 0.92, [6, 9], 0.8200000000000001, [7, 8], 0.91, [7, 9], 0.14, [8, 9], 0.39]",27,"[new_layer_p, 10, -0.5, 0.35000000000000003, new_layer_p, 10, -0.37, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.14, 1.26]",-9.563417,-9.96
0,er_graph_0,"[[1, 3], 0.46, [1, 5], 0.39, [1, 6], 0.35000000000000003, [2, 3], 0.76, [2, 4], 0.31, [2, 5], 0.53, [2, 6], 0.45, [2, 7], 0.31, [2, 9], 0.1, [3, 4], 0.46, [3, 5], 0.72, [3, 6], 0.34, [3, 7], 0.79, [3, 8], 0.4, [4, 5], 0.08, [4, 6], 0.24, [4, 7], 0.77, [4, 8], 0.59, [4, 9], 0.9400000000000001, [5, 8], 0.29, [5, 9], 0.7000000000000001, [6, 7], 0.45, [6, 8], 0.92, [6, 9], 0.8200000000000001, [7, 8], 0.91, [7, 9], 0.14, [8, 9], 0.39]",27,"[new_layer_p, 10, -0.5, 0.35000000000000003, new_layer_p, 10, -0.36, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.32, 0.78, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.21, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.14, 1.13, new_layer_p, 10, -0.09, 1.26]",-9.543285,-9.96


In [20]:
qaoa_gpt_circ_eval_expl_df[qaoa_gpt_circ_eval_expl_df['adapt_gpt_energies'] > 0][:2]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi


This computes how close each QAOA-GPT circuit’s energy is to the optimal solution (AR — approximation ratio).

If the ratio = 1 → perfect solution.

If <1 → circuit energy is above the optimal energy (since MaxCut is a minimization problem in some conventions, sometimes they flip it; conceptually, ratio shows performance).

In [21]:
(qaoa_gpt_circ_eval_expl_df['adapt_gpt_energies'] / qaoa_gpt_circ_eval_expl_df['energy_gurobi']).mean()

np.float64(0.9406987969280451)

In [22]:
# on avg, how many layers are there in the predicted circuits
qaoa_gpt_circ_eval_expl_df['q_circuits'].apply(lambda x: x.count('new_layer_p')).mean()

np.float64(9.8)

The code above is incorrect becasue it not count the error circuits which have adapt_gpt_energies = 999, so you shoul used compute_metrics

In [23]:
ar, err, layers = compute_metrics(qaoa_gpt_circ_eval_df)
print(f"Avg AR: {ar}, Avg ERR: {err}, Avg Layers: {layers}")

Avg AR: 0.9406987969280454, Avg ERR: 0.0, Avg Layers: 9.8
